# 03 领域术语总混淆？教你构建精准术语词库，提升检索一致性

在RAG系统构建过程中，术语混淆直接影响信息检索的精准度与生成内容的质量。

这主要源于几个方面：
- 向量表示
- 不同行业、公司乃至同一组织内部，都可能存在相似词汇却拥有截然不同含义的情况

这些因素最终导致检索结果偏离预期，大幅降低了答案的质量。

# 一、术语词库构建与维护（Glossary Management）
## 1.1 产生术语混淆的原因

- 术语多义性
- 同义词与近义词
- 领域差异
- 企业专属术语

## 1.2 构建术语词库的目标
术语词库是整个术语一致性优化体系的核心基础设施。

## 1.3 术语词库的构建流程
- Step 1：收集术语来源
- Step 2：标准化术语
- Step 3：建立别名映射关系
- Step 4：添加上下文信息
- Step 5：构建术语索引


一个功能完善的术语词库应包含以下关键字段，以确保其结构化和可操作性：

| 字段名                 | 内容                                                                                   |
|------------------------|----------------------------------------------------------------------------------------|
| 术语（Term）           | 神经网络                                                                              |
| 别名（Synonyms）       | ["人工神经网络", "NN"]                                                                |
| 定义（Definition）     | 神经网络是一种模仿生物神经网络结构和功能的计算模型……                                  |
| 上下文标签（Context Tags） | ["人工智能", "深度学习", "计算机科学"]                                                 |
| 所属领域（Domain）     | 人工智能                                                                              |
| 示例用法（Usage Example） | 在图像识别任务中，我们使用了一个卷积神经网络。                                        |
| 外部链接（External Link） | [维基百科链接](https://en.wikipedia.org/wiki/Artificial_neural_network)               |
| 禁用词/误导词（Stop Words / Misleading Terms） | ["神经系统"（医学中的不同概念）]                                                  |

## 1.4 术语词库与 RAG 集成

- 方式一：预处理阶段替换术语
- 方式二：检索增强
- 方式三：重排序（Re-ranking）
- 方式四：后处理解释

## 1.5 术语词库维护
1. 术语词库结构设计
这是基础，确定词库所需包含的字段和它们之间的关系。

2. 自动抽取术语候选
利用 NLP 工具从大量文本中自动识别和提取潜在的术语。

3. 专家审核与完善
领域专家对自动抽取的术语进行人工审核、修正和补充，确保准确性和专业性。

4. 构建术语关系图谱
如果有需求，可以进一步构建术语之间的层次、关联关系，形成本体（Ontology）或知识图谱（Knowledge Graph），以提升语义理解能力。

5. 版本控制与更新机制建设
建立术语词库的版本管理和定期更新机制，确保其时效性和权威性，应对新术语的出现或旧术语含义的变化。

| 阶段             | 技术名称                                 | 描述                                                                                                                                               |
|------------------|------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------|
| 1. 数据预处理    | 术语抽取、标准化、上下文分块             | 在原始文档和查询进入 RAG 系统之前，识别并提取领域术语，进行统一化处理，并确保文本分块时能有效保留术语的上下文信息。                                   |
| 2. 术语词库构建  | 词库设计、术语关系建模、版本管理         | 建立结构化的术语词库，包含术语、别名、定义、上下文标签等字段。进一步可构建术语间的层级或关联关系（如本体），并建立完善的版本控制与更新机制。           |
| 3. 嵌入与向量化  | 构建术语向量索引、微调领域嵌入模型       | 将术语词库中的标准术语和别名转换为向量，并构建高效的向量索引（如 FAISS）。同时，通过领域适应性训练（如 LoRA）优化通用嵌入模型，使其更好地理解领域特有概念。 |
| 4. 检索增强      | 查询扩展、混合检索、重排序、元数据过滤   | 利用术语词库对用户查询进行扩展（添加别名），结合向量检索与关键词检索（混合检索）。在召回结果后，通过术语匹配度进行重排序，或利用术语作为元数据进行更精确的过滤。 |
| 5. 生成控制      | 提示工程、结构化输出、术语验证           | 设计包含术语词库信息的提示词，引导大模型生成更准确的答案。在输出阶段，可强制模型使用词库中的标准术语，并对生成内容进行术语验证，避免出现混淆或不规范表达。     |
| 6. 评估与反馈    | 术语一致性指标、LLM-as-a-Judge、用户反馈 | 建立专门的评估指标来衡量 RAG 系统在术语一致性方面的表现。利用大型语言模型作为评估器（LLM-as-a-Judge）来检查术语使用情况，并收集用户反馈，持续优化词库和系统。 |

## demo

In [6]:
import re
from typing import List, Dict, Any

# --- 1. 定义术语词库 (数据驱动的核心) ---
# 每个条目包含标准术语、同义词、定义以及用于消歧的上下文标签
GLOSSARY = [
    {
        "term": "卷积神经网络", 
        "synonyms": ["CNN", "卷基神经网络"], 
        "definition": "一种专门处理图像的深度学习模型。", 
        "context_tags": ["图像识别", "计算机视觉", "深度学习"]
    },
    {
        "term": "机器学习", 
        "synonyms": ["ML", "AI算法"], 
        "definition": "人工智能的一个子集，侧重于数据学习。", 
        "context_tags": ["人工智能", "数据科学", "预测"]
    },
    {
        "term": "中央处理器", 
        "synonyms": ["CPU"], 
        "definition": "计算机的运算核心。", 
        "context_tags": ["计算机硬件", "主板", "性能", "电脑"]
    },
    {
        "term": "每单位成本", 
        "synonyms": ["CPU"], 
        "definition": "业务分析中衡量每个产品单位的平均成本。", 
        "context_tags": ["财务", "业务分析", "利润", "成本", "报表"]
    },
]

# --- 2. 定义术语处理器类 ---
class TerminologyProcessor:
    def __init__(self, glossary: List[Dict[str, Any]]):
        self.glossary = glossary
        self.standard_term_map = {} # 用于快速查找标准术语
        self.alias_to_entries_map = {} # 用于处理同义词到多个定义的映射（消歧）
        self._build_mappings()

    def _build_mappings(self):
        """预处理词库，构建双向索引映射。"""
        for entry in self.glossary:
            standard_term = entry["term"]
            # 建立标准术语到条目的映射
            self.standard_term_map[standard_term.lower()] = standard_term

            # 将标准术语本身也视作一个“别名”，方便统一处理
            all_aliases = [standard_term] + entry.get("synonyms", [])
            # 建立同义词到条目的映射
            for alias in all_aliases:
                alias_lower = alias.lower()
                if alias_lower not in self.alias_to_entries_map:
                    self.alias_to_entries_map[alias_lower] = []
                if entry not in self.alias_to_entries_map[alias_lower]:
                    self.alias_to_entries_map[alias_lower].append(entry)

    def standardize_text(self, text: str, context_window: int = 10) -> str:
        """
        核心功能：文本标准化。
        遍历所有已知别名，按长度倒序匹配（长词优先），并进行上下文消歧。
        """
        if not text:
            return ""
        
        standardized_text = text
        # 按长度降序排列 key，防止“卷积神经网络”被拆解匹配成“网络”
        sorted_keys = sorted(self.alias_to_entries_map.keys(), key=len, reverse=True)
        
        for key_lower in sorted_keys:
            possible_entries = self.alias_to_entries_map[key_lower]

            # 动态构建正则表达式
            # 如果是英文缩写，增加单词边界判断 (?<![a-zA-Z]) 防止误伤单词内部字母
            if re.search(r'[a-zA-Z]', key_lower):
                pattern_str = r'(?<![a-zA-Z])' + re.escape(key_lower) + r'(?![a-zA-Z])'
            else:
                pattern_str = re.escape(key_lower)

            pattern = re.compile(pattern_str, flags=re.IGNORECASE)

            # 内部替换函数：决定每一个 match 应该替换成哪个标准术语
            def replacer(match: re.Match) -> str:
                # 场景 A: 只有一个候选定义，直接替换
                if len(possible_entries) == 1:
                    return possible_entries[0]["term"]
                
                # 场景 B: 有多个候选（消歧逻辑）
                # 截取匹配点前后的文本块作为“上下文”
                start_idx = max(0, match.start() - context_window)
                end_idx = min(len(standardized_text), match.end() + context_window)
                context = standardized_text[start_idx:end_idx]

                for entry in possible_entries:
                    # 检查上下文是否包含该术语的特征标签
                    clues = entry.get("context_tags", []) + [entry["term"]]
                    if any(clue in context for clue in clues):
                        return entry["term"]
                # 若无任何线索，默认返回第一个定义
                return possible_entries[0]["term"]

            # 执行正则替换
            standardized_text = pattern.sub(replacer, standardized_text)
        return standardized_text
    
    def extract_terms(self, text: str) -> List[str]:
        """从文本中提取已知的标准术语"""
        found_terms = set()
        text_lower = text.lower()

        for standard_term_lower, original_standard_term in self.standard_term_map.items():
            if re.search(re.escape(standard_term_lower), text_lower):
                found_terms.add(original_standard_term)
        return sorted(list(found_terms))
    

# 1. 使用词库初始化术语处理器。
term_processor = TerminologyProcessor(GLOSSARY)

print(term_processor.standard_term_map)
# 2. 数据预处理阶段：术语标准化
print("--- 2. 数据预处理阶段：术语标准化 ---")
user_query = "我想了解一下ML模型在图像识别中的应用，还有NLP的相关知识。"
processed_query = term_processor.standardize_text(user_query)
print(f"原始查询: {user_query}")
print(f"标准化查询: {processed_query}")


document_text = "最近研究了CNN和AI算法，发现它们在处理大数据方面表现出色，特别是ML在某些场景下的应用。"
processed_document = term_processor.standardize_text(document_text)
print(f"原始文档: {document_text}")
print(f"标准化文档: {processed_document}")

# 3. 术语提取（用于后续的向量化或元数据标记）
print("\n--- 3. 术语提取 ---")
extracted_terms_query = term_processor.extract_terms(processed_query)
print(f"从查询中提取的术语: {extracted_terms_query}")

extracted_terms_document = term_processor.extract_terms(processed_document)
print(f"从文档中提取的术语: {extracted_terms_document}")

# 4. 模拟向量化存储和检索增强（概念性）
print("\n--- 4. 模拟向量化存储和检索增强（概念性） ---")
print("在实际应用中，我们将使用嵌入模型（例如 SentenceTransformers）将标准化后的文本和术语转换为向量。")
print("这些向量随后存储在专门的向量数据库中（如 FAISS、Pinecone 或 Weaviate），以便进行高效的相似性搜索。")
print("在检索过程中，用户查询首先被标准化并向量化，然后用于查询向量数据库以获取相关文档。")

def enhance_query_for_retrieval(qeury: str, processor: TerminologyProcessor) -> List[str]:
    """通过术语词库扩展查询关键词以提高召回率。"""
    standardized_query = processor.standardize_text(qeury)
    query_terms = processor.extract_terms(standardized_query)

    expanded_keywords = set([standardized_query])
    for term in query_terms:
        expanded_keywords.add(term)
        for entry in processor.glossary:
            if entry["term"] == term:
                for symonym in entry.get("synonyms", []):
                    expanded_keywords.add(symonym)
                    break
    return sorted(list(expanded_keywords))

print("\n--- 5. 模拟检索增强：查询扩展 ---")
original_query_for_retrieval = "我想知道CPU在电脑里的作用，还有成本CPU是什么？"
expanded_keywords = enhance_query_for_retrieval(original_query_for_retrieval, term_processor)
print(f"原始检索查询: {original_query_for_retrieval}")
print(f"扩展后的检索关键词列表: {expanded_keywords}")

print("\n在生产RAG系统中，这些扩展关键词将驱动混合检索策略，结合语义（向量）搜索与基于关键词的搜索，以获得最佳结果。")

{'卷积神经网络': '卷积神经网络', '机器学习': '机器学习', '中央处理器': '中央处理器', '每单位成本': '每单位成本'}
--- 2. 数据预处理阶段：术语标准化 ---
原始查询: 我想了解一下ML模型在图像识别中的应用，还有NLP的相关知识。
标准化查询: 我想了解一下机器学习模型在图像识别中的应用，还有NLP的相关知识。
原始文档: 最近研究了CNN和AI算法，发现它们在处理大数据方面表现出色，特别是ML在某些场景下的应用。
标准化文档: 最近研究了卷积神经网络和机器学习，发现它们在处理大数据方面表现出色，特别是机器学习在某些场景下的应用。

--- 3. 术语提取 ---
从查询中提取的术语: ['机器学习']
从文档中提取的术语: ['卷积神经网络', '机器学习']

--- 4. 模拟向量化存储和检索增强（概念性） ---
在实际应用中，我们将使用嵌入模型（例如 SentenceTransformers）将标准化后的文本和术语转换为向量。
这些向量随后存储在专门的向量数据库中（如 FAISS、Pinecone 或 Weaviate），以便进行高效的相似性搜索。
在检索过程中，用户查询首先被标准化并向量化，然后用于查询向量数据库以获取相关文档。

--- 5. 模拟检索增强：查询扩展 ---
原始检索查询: 我想知道CPU在电脑里的作用，还有成本CPU是什么？
扩展后的检索关键词列表: ['CPU', '中央处理器', '我想知道中央处理器在电脑里的作用，还有成本每单位成本是什么？', '每单位成本']

在生产RAG系统中，这些扩展关键词将驱动混合检索策略，结合语义（向量）搜索与基于关键词的搜索，以获得最佳结果。
